# 🏋️ Entrenamiento YOLO26 - Trabajo Final Visión por Computadora
## Detección, Segmentación y Estimación de Pose

Este notebook entrena un modelo **YOLO26** con clases personalizadas (no incluidas en COCO) usando **transfer learning**.

### 📋 Configuración del proyecto
- **Clases**: mate, termo, factura (Set A por defecto)
- **Imágenes por clase**: ≥50 (40 train + 10 val)
- **Épocas**: 80 (ajustable)
- **Batch size**: 16 (ajustar según VRAM)
- **Image size**: 640

> ⚠️ **IMPORTANTE**: Si tu versión de `ultralytics` no tiene `yolo26n.pt`, reemplazá por la versión disponible (`yolo11n.pt`, `yolo12n.pt`, etc.).

## 1. 📦 Instalación de dependencias

In [ ]:
!pip install -q ultralytics opencv-python numpy torch torchvision Pillow PyYAML

## 2. 🖥️ Verificar GPU

Es importante contar con GPU para entrenar en tiempos razonables. Si no tenés GPU local, considerá usar Google Colab o Kaggle (ambos con GPU gratuita).

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")
else:
    print("⚠️ No hay GPU disponible. El entrenamiento será lento. Considerá usar Colab/Kaggle.")

## 3. 📁 Verificar el dataset

Asegurate de tener el dataset armado en `data/images/{train,val}/` y `data/labels/{train,val}/` con el `data.yaml` correspondiente.

In [ ]:
import os
from pathlib import Path

# Mostrar contenido de data.yaml
with open('data/data.yaml', 'r', encoding='utf-8') as f:
    print("=== data.yaml ===")
    print(f.read())

# Contar imagenes
data_dir = Path('data')
for split in ['train', 'val']:
    img_dir = data_dir / 'images' / split
    lbl_dir = data_dir / 'labels' / split
    n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f"{split}: {n_imgs} imagenes, {n_lbls} labels")

## 4. 🧠 Cargar modelo preentrenado (Transfer Learning)

Elegí el tamaño del modelo según tu VRAM disponible:
- `yolo26n.pt` (nano) - más rápido, menos VRAM
- `yolo26s.pt` (small) - balance
- `yolo26m.pt` (medium) - más precisión, más VRAM

In [ ]:
from ultralytics import YOLO

# Si tu ultralytics no tiene yolo26, cambiar por la version disponible:
# yolo11n.pt, yolo12n.pt, yolo13n.pt, etc.
MODEL_NAME = 'yolo26n.pt'  # <-- Cambiar aca si no existe

model = YOLO(MODEL_NAME)
print(f"Modelo {MODEL_NAME} cargado correctamente")
print("Tareas disponibles: detect, segment, classify, pose")

## 5. 🚀 Entrenamiento

Ajustá los hiperparámetros según tu hardware. El entrenamiento se guarda automáticamente en `runs/detect/train/`.

In [ ]:
results = model.train(
    data='data/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    patience=15,           # early stopping
    augment=True,          # augmentations automaticas
    project='runs/detect',
    name='train',
    exist_ok=True,
    plots=True,            # guardar graficas
    save=True,
)

## 6. 📊 Validación y métricas

Evalúa el modelo entrenado sobre el conjunto de validación.

In [ ]:
metrics = model.val()

print("=" * 50)
print("MÉTRICAS DEL MODELO ENTRENADO")
print("=" * 50)
print(f"mAP50:       {metrics.box.map50:.4f}")
print(f"mAP50-95:    {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")
print()
print("Las graficas (loss, matriz confusion, PR) estan en:")
print("  runs/detect/train/")

## 7. 🖼️ Test con imagen de validación

Corre predicciones sobre imágenes del set de validación y guarda los resultados para inspección visual.

In [ ]:
from PIL import Image

test_results = model.predict(
    source='data/images/val',
    conf=0.4,
    save=True,
    project='runs/detect',
    name='predict_test',
    exist_ok=True,
)

print(f"Predicciones guardadas en runs/detect/predict_test/")
print(f"Total de imágenes procesadas: {len(test_results)}")

## 8. 💾 Copiar best.pt al directorio del proyecto

El archivo `best.pt` es el entregable principal del entrenamiento (los pesos con mejor mAP).

In [ ]:
import shutil
from pathlib import Path

src = Path('runs/detect/train/weights/best.pt')
dst = Path('best.pt')

if src.exists():
    shutil.copy(src, dst)
    print(f"✅ best.pt copiado a: {dst.resolve()}")
    print(f"   Tamaño: {dst.stat().st_size / 1e6:.2f} MB")
else:
    print(f"❌ No se encontró {src}")
    print("Verificá que el entrenamiento haya finalizado correctamente.")